# AI-Powered Sentiment Analysis System

**Month 2 Project | Natural Language Processing + Machine Learning**

This notebook implements the complete workflow required in the project brief: data validation, exploratory analysis, text preprocessing, Bag-of-Words/TF-IDF feature extraction, model comparison, grouped cross-validation, hyperparameter tuning, final evaluation and model serialization.

**Target:** `sentiment` with three classes: Positive, Negative and Neutral.

> Data-quality warning: the supplied file contains 10,000 rows but only 300 distinct review texts. Identical review texts are grouped during splitting and cross-validation to prevent text leakage. The resulting scores describe performance on this supplied, repetitive corpus and should not be generalized to arbitrary customer reviews.

## 1. Import libraries and locate the project

In [ ]:
from pathlib import Path
import sys, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix, roc_auc_score,
                             RocCurveDisplay)
from sklearn.preprocessing import label_binarize
import joblib

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from sentiment_utils import clean_text
DATA_PATH = ROOT / 'data' / 'reviews.csv'
RANDOM_STATE = 42
sns.set_theme(style='whitegrid')

## 2. Load and understand the dataset

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
print('Shape:', df_raw.shape)
display(df_raw.head())
print('Data types:')
display(df_raw.dtypes.to_frame('dtype'))
print('Missing values:')
display(df_raw.isna().sum().to_frame('missing_count'))
print('Exact duplicate rows:', df_raw.duplicated().sum())
print('Unique review texts:', df_raw['review_text'].nunique())
print('Repeated rows beyond unique text count:', len(df_raw) - df_raw['review_text'].nunique())

In [ ]:
# Validate required fields and clean only genuinely invalid rows / exact duplicate records.
required = {'review_id', 'review_text', 'sentiment', 'category', 'source'}
assert required.issubset(df_raw.columns)
df = df_raw.copy()
df['review_text'] = df['review_text'].fillna('').astype(str)
df['sentiment'] = df['sentiment'].astype(str).str.strip().str.title()
df = df[df['review_text'].str.strip().ne('')].drop_duplicates().reset_index(drop=True)
classes = ['Negative', 'Neutral', 'Positive']
assert set(df['sentiment'].unique()) == set(classes)
print('Rows after cleaning:', len(df))
display(df['sentiment'].value_counts().rename_axis('sentiment').to_frame('count'))

## 3. Exploratory Data Analysis

In [ ]:
df['word_count'] = df['review_text'].str.split().str.len()
print('Mean words:', round(df['word_count'].mean(), 2))
print('Median words:', df['word_count'].median())
fig, ax = plt.subplots(figsize=(7, 4))
df['sentiment'].value_counts().reindex(classes).plot(kind='bar', ax=ax)
ax.set(title='Sentiment Class Distribution', xlabel='Sentiment', ylabel='Number of reviews'); plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(data=df, x='word_count', hue='sentiment', bins=25, element='step', stat='count', common_norm=False, ax=ax)
ax.set(title='Review Length by Sentiment', xlabel='Words per review'); plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(8, 4))
df['category'].value_counts().plot(kind='barh', ax=ax)
ax.set(title='Reviews by Category', xlabel='Number of reviews'); plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(8, 4))
df['source'].value_counts().plot(kind='barh', ax=ax)
ax.set(title='Reviews by Source', xlabel='Number of reviews'); plt.tight_layout(); plt.show()

## 4. Text preprocessing and feature extraction

The normalizer removes HTML markup, normalizes URLs, converts text to lowercase, removes most punctuation and collapses whitespace. It deliberately retains negation terms (`not`, `no`) because they can reverse sentiment. TF-IDF downweights terms that occur in many documents; Bag of Words records token counts.

In [ ]:
examples = [
    'This product is NOT good!!! Visit https://example.com',
    '<b>Excellent</b> quality &amp; very easy to use.'
]
for text in examples:
    print(text, ' -> ', clean_text(text))

# Small illustration: vectorizers are fit only on training text later.
illustration = ['good product', 'bad product', 'good quality']
print('BoW vocabulary:', CountVectorizer().fit(illustration).get_feature_names_out())
print('TF-IDF vocabulary:', TfidfVectorizer().fit(illustration).get_feature_names_out())

## 5. Leakage-aware train/test split

A standard random row split would place repeated copies of the same review text on both sides. Instead, `StratifiedGroupKFold` keeps every identical review text in a single partition while approximately preserving class proportions.

In [ ]:
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(df['review_text'], df['sentiment'], groups=df['review_text']))
X_train, X_test = df['review_text'].iloc[train_idx], df['review_text'].iloc[test_idx]
y_train, y_test = df['sentiment'].iloc[train_idx], df['sentiment'].iloc[test_idx]
train_groups = X_train.values
print('Train rows:', len(X_train), 'Test rows:', len(X_test))
print('Train classes:'); display(y_train.value_counts())
print('Test classes:'); display(y_test.value_counts())
assert set(X_train).isdisjoint(set(X_test)), 'Identical review text leakage detected!'

## 6. Train and compare baseline classifiers

The vectorizer is inside each pipeline, so vocabulary learning occurs only on the relevant training fold. Macro precision, recall and F1 give each class equal weight.

In [ ]:
models = {
    'MultinomialNB (TF-IDF)': Pipeline([('tfidf', TfidfVectorizer(preprocessor=clean_text, ngram_range=(1, 2), min_df=2)), ('clf', MultinomialNB())]),
    'Logistic Regression (TF-IDF)': Pipeline([('tfidf', TfidfVectorizer(preprocessor=clean_text, ngram_range=(1, 2), min_df=2)), ('clf', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))]),
    'Linear SVM (TF-IDF)': Pipeline([('tfidf', TfidfVectorizer(preprocessor=clean_text, ngram_range=(1, 2), min_df=2)), ('clf', LinearSVC(random_state=RANDOM_STATE))]),
    'Logistic Regression (BoW)': Pipeline([('bow', CountVectorizer(preprocessor=clean_text, ngram_range=(1, 2), min_df=2)), ('clf', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))])
}
rows = []
for name, model in models.items():
    start = time.time(); model.fit(X_train, y_train); pred = model.predict(X_test)
    rows.append({'Model': name, 'Accuracy': accuracy_score(y_test, pred),
                 'Precision_macro': precision_score(y_test, pred, average='macro'),
                 'Recall_macro': recall_score(y_test, pred, average='macro'),
                 'F1_macro': f1_score(y_test, pred, average='macro'),
                 'Training_seconds': time.time() - start})
comparison = pd.DataFrame(rows).sort_values('F1_macro', ascending=False)
display(comparison)

## 7. Hyperparameter tuning with grouped cross-validation

Grid Search selects the regularization strength `C`, n-gram range and minimum document frequency using macro F1. Five-fold stratified group cross-validation ensures duplicate texts are not shared between folds.

In [ ]:
base_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(preprocessor=clean_text)),
    ('clf', LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))
])
param_grid = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__min_df': [1, 2],
    'clf__C': [0.5, 1.0, 2.0]
}
group_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
grid = GridSearchCV(base_pipeline, param_grid, scoring='f1_macro', cv=group_cv,
                    n_jobs=-1, refit=True, return_train_score=True)
grid.fit(X_train, y_train, groups=train_groups)
best_model = grid.best_estimator_
print('Best parameters:', grid.best_params_)
print('Best CV macro F1:', grid.best_score_)

## 8. Final model evaluation on the untouched test set

In [ ]:
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)
print(classification_report(y_test, y_pred, labels=classes, zero_division=0))
final_metrics = {
    'accuracy': accuracy_score(y_test, y_pred),
    'precision_macro': precision_score(y_test, y_pred, average='macro'),
    'recall_macro': recall_score(y_test, y_pred, average='macro'),
    'f1_macro': f1_score(y_test, y_pred, average='macro'),
    'roc_auc_ovr_macro': roc_auc_score(y_test, y_prob, multi_class='ovr', average='macro')
}
print(final_metrics)
cm = confusion_matrix(y_test, y_pred, labels=classes)
fig, ax = plt.subplots(figsize=(6, 5)); sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes, ax=ax)
ax.set(title='Confusion Matrix', xlabel='Predicted', ylabel='Actual'); plt.tight_layout(); plt.show()
Y_bin = label_binarize(y_test, classes=classes)
fig, ax = plt.subplots(figsize=(6, 5))
for i, label in enumerate(classes):
    RocCurveDisplay.from_predictions(Y_bin[:, i], y_prob[:, i], name=label, ax=ax)
ax.set_title('Multiclass ROC Curves (One-vs-Rest)'); plt.tight_layout(); plt.show()

## 9. Cross-validation and overfitting diagnostic

In [ ]:
cv_scores = cross_validate(best_model, X_train, y_train, groups=train_groups, cv=group_cv,
                           scoring={'accuracy':'accuracy', 'precision_macro':'precision_macro',
                                    'recall_macro':'recall_macro', 'f1_macro':'f1_macro'}, n_jobs=-1)
cv_means = {metric: float(np.mean(values)) for metric, values in cv_scores.items() if metric.startswith('test_')}
cv_means = {k.replace('test_', ''): v for k, v in cv_means.items()}
best_row = grid.cv_results_['mean_train_score'][grid.best_index_]
best_validation = grid.cv_results_['mean_test_score'][grid.best_index_]
print('Mean CV metrics:', cv_means)
print('Best candidate train-fold macro F1:', best_row)
print('Best candidate validation macro F1:', best_validation)
print('Train-validation gap:', best_row - best_validation)
print('Interpretation: a small gap suggests limited overfitting on these folds; it does not prove performance on external data.')

## 10. Save the complete pipeline

In [ ]:
model_dir = ROOT / 'models'
model_dir.mkdir(exist_ok=True)
joblib.dump(best_model, model_dir / 'sentiment_pipeline.joblib')
joblib.dump(best_model, model_dir / 'sentiment_pipeline.pkl')
print('Saved:', model_dir / 'sentiment_pipeline.joblib')

## 11. Test inference on new text

In [ ]:
new_reviews = [
    'I absolutely love this product, it works perfectly.',
    'This is disappointing and does not work as expected.',
    'The package contains a charger and a user manual.'
]
new_predictions = best_model.predict(new_reviews)
new_probabilities = best_model.predict_proba(new_reviews)
for text, label, probs in zip(new_reviews, new_predictions, new_probabilities):
    print(f'Review: {text}')
    print(f'Predicted: {label} | estimated probability: {probs.max():.2%}')

## 12. Conclusion and limitations

This notebook demonstrates a complete supervised NLP workflow. The selected pipeline combines text normalization, TF-IDF and Logistic Regression, with model selection performed using grouped cross-validation. The held-out metrics are extremely high, but the dataset contains only 300 distinct review strings repeated many times. This means the model is largely learning a limited collection of repeated templates. For credible real-world deployment, obtain a substantially larger and more varied set of independently written reviews, label them carefully, and evaluate on a separate external dataset. The Streamlit app should be treated as an educational prototype.